# Fund Reporting Pipeline – Robeco 3D ETFs (unofficial replication)

This notebook replicates a monthly fund reporting workflow with public data:
download prices → store raw data → run data quality checks → build a clean price table
→ calculate performance → reconcile against published figures → export to Excel.

- **Report date:** 30 September 2026
- **Data:** Xetra closing prices from Yahoo Finance, snapshot downloaded on 7 November 2026
- **Funds:** Robeco 3D Global Equity (3DGE), 3D EUR Enhanced Index Credits (3DCE), 3D Global Enhanced Index Credits (3DCH)
- **Market reference:** iShares Core MSCI World (EUNL). Used for context only, not as a benchmark.

*Built for learning purposes. Not affiliated with or endorsed by Robeco.*

In [31]:
import os
import yfinance as yf
import sqlite3
import pandas as pd
import exchange_calendars as xcals

## 1. Download prices and save a snapshot

Prices are downloaded once and saved to `data/raw/` so that results can be reproduced
even if the data source changes later.

In [32]:
os.makedirs("data/raw", exist_ok=True)   # Create the folder; no error if it already exists
tickers = ["3DGE.DE", "3DCE.DE", "3DCH.DE", "EUNL.DE"]
data = yf.download(tickers, start="2025-05-01", end="2026-11-07", threads=False)["Close"]
data.to_csv("data/raw/prices_2026-11-07.csv")   # Save a snapshot; always read from this file from now on
print(data.shape)          
print(data.isna().sum())   
data.head()

[*********************100%***********************]  4 of 4 completed

(387, 4)
Ticker
3DCE.DE    265
3DCH.DE    190
3DGE.DE      0
EUNL.DE      1
dtype: int64


Ticker,3DCE.DE,3DCH.DE,3DGE.DE,EUNL.DE
Date,,,,
2025-05-02,NaN,NaN,5.150,95.648003
2025-05-05,NaN,NaN,5.130,95.781998
2025-05-06,NaN,NaN,5.100,95.279999
2025-05-07,NaN,NaN,5.137,94.825996
2025-05-08,NaN,NaN,5.152,96.365997


## 2. First look at the data

Missing values before a fund's first price are expected (the fund was not listed yet).
Missing values after listing are potential data issues and are checked in section 4.

In [33]:
# First date with data for each fund = listing date
print(data.apply(lambda s: s.first_valid_index()))

# Which day is EUNL missing?
data[data["EUNL.DE"].isna()]

Ticker
3DCE.DE   2026-05-21
3DCH.DE   2026-02-02
3DGE.DE   2025-05-02
EUNL.DE   2025-05-02
dtype: datetime64[ns]


Ticker,3DCE.DE,3DCH.DE,3DGE.DE,EUNL.DE
Date,,,,
2026-03-06,NaN,5.0,6.191,NaN


In [34]:
# Report date: 2026-09-30
asof = "2026-09-30"
report = data.loc[:asof]          
print(report.tail())              # Confirm the last date is 30 Sep 2026

Ticker      3DCE.DE  3DCH.DE  3DGE.DE     EUNL.DE
Date                                             
2026-09-24   4.9505   4.8865    6.940  128.350006
2026-09-25   4.9508   4.8858    6.998  128.884995
2026-09-28   4.9430   4.8683    6.965  128.559998
2026-09-29   4.9448   4.8635    6.955  128.705002
2026-09-30   4.9603   4.8818    6.993  129.134995


## 3. Load raw prices into SQLite

Raw prices are stored unchanged in `prices_raw`. Reports are never built directly from raw data:
they are built from `prices_clean` (section 5) after quality checks.
Row counts here reconcile with the source file.

In [35]:
# 1. Wide to long: one (fund, date, price) per row
prices_long = (
    data.reset_index()
        .melt(id_vars="Date", var_name="ticker", value_name="close")
        .dropna(subset=["close"])          	# Drop blanks before each fund's listing
        .rename(columns={"Date": "date"})
)
prices_long["date"] = prices_long["date"].dt.strftime("%Y-%m-%d")

# 2. Create the database and store raw prices unchanged
conn = sqlite3.connect("fund_reporting.db")
prices_long.to_sql("prices_raw", conn, if_exists="replace", index=False)

# 3. Row count and date range per fund
pd.read_sql("""
    SELECT ticker,
           COUNT(*)  AS n_days,
           MIN(date) AS first_date,
           MAX(date) AS last_date
    FROM prices_raw
    GROUP BY ticker
""", conn)

,ticker,n_days,first_date,last_date
0,3DCE.DE,122,2026-05-21,2026-11-06
1,3DCH.DE,197,2026-02-02,2026-11-06
2,3DGE.DE,387,2025-05-02,2026-11-06
3,EUNL.DE,386,2025-05-02,2026-11-06


## 4. Data quality checks

Each fund is checked against the official Xetra trading calendar from its listing date.
Using an independent calendar matters: a day missing for every fund would not be visible
if the calendar were built from the data itself.

**Findings**

| Check | Result | Impact | Action |
|---|---|---|---|
| Missing days | 24 Oct 2025: no price for any fund | None (not a month-end) | Forward-fill, flagged |
| Missing days | 6 Mar 2026: no price for EUNL | None (not a month-end, reference only) | Forward-fill, flagged |
| Prices on closed days | None | – | – |
| Duplicates | None | – | – |
| Outliers (> 4 std. dev.) | 5 days | 4 match market moves; 3DGE 13 May 2025 (+3.7% vs +0.6% market) is likely a lagged price in a newly listed ETF, not verified. None fall on a month-end, so reported returns are unaffected | Documented |

In [36]:
xetr = xcals.get_calendar("XETR")
sessions = xetr.sessions_in_range("2025-05-02", "2026-11-06")
calendar = pd.DataFrame({"date": sessions.strftime("%Y-%m-%d")})
calendar.to_sql("calendar", conn, if_exists="replace", index=False)

388

In [37]:
# Missing days: compare each fund with the official calendar from its listing date
pd.read_sql("""
    WITH listing AS (
        SELECT ticker, MIN(date) AS first_date
        FROM prices_raw
        GROUP BY ticker
    )
    SELECT l.ticker, c.date AS missing_date
    FROM calendar AS c
    JOIN listing AS l
         ON c.date >= l.first_date          -- Only dates after listing
    LEFT JOIN prices_raw AS p
         ON p.ticker = l.ticker
        AND p.date   = c.date
    WHERE p.date IS NULL
      AND c.date <= '2026-09-30'            -- Only dates within the report period
    ORDER BY l.ticker, c.date
""", conn)

,ticker,missing_date
0,3DGE.DE,2025-10-24
1,EUNL.DE,2025-10-24
2,EUNL.DE,2026-03-06


In [38]:
# Trading days on the official calendar where no fund has a price
pd.read_sql("""
    SELECT c.date
    FROM calendar AS c
    LEFT JOIN (SELECT DISTINCT date FROM prices_raw) AS p
           ON p.date = c.date
    WHERE p.date IS NULL
""", conn)



,date
0,2025-10-24


In [39]:
# Reverse check: prices on days the exchange was officially closed
pd.read_sql("""
    SELECT DISTINCT p.date
    FROM prices_raw AS p
    LEFT JOIN calendar AS c ON c.date = p.date
    WHERE c.date IS NULL
""", conn)

,date


In [40]:
# Daily returns in SQL, standard deviation in pandas 
daily = pd.read_sql("""
    SELECT ticker, date,
           close / LAG(close) OVER (PARTITION BY ticker ORDER BY date) - 1 AS ret
    FROM prices_raw
""", conn).dropna()

# Flag days more than 4 standard deviations from the fund's own average
daily["z"] = daily.groupby("ticker")["ret"].transform(lambda r: (r - r.mean()) / r.std())
outliers = daily[daily["z"].abs() > 4]



In [41]:
# Compare with the market reference (EUNL) on the same day
market_ref = daily[daily["ticker"] == "EUNL.DE"][["date", "ret"]].rename(columns={"ret": "market_ref_ret"})
outliers.merge(market_ref, on="date", how="left")

,ticker,date,ret,z,market_ref_ret
0,3DCE.DE,2026-05-25,0.007349,4.022907,0.005291
1,3DGE.DE,2025-05-13,0.036611,4.792018,0.006091
2,3DGE.DE,2026-04-08,0.034136,4.461511,0.025199
3,EUNL.DE,2025-05-12,0.032870,4.522589,0.032870
4,EUNL.DE,2025-08-01,-0.030146,-4.355203,-0.030146


In [45]:
pd.read_sql("""
    SELECT ticker, date, COUNT(*) AS n
    FROM prices_raw
    GROUP BY ticker, date
    HAVING COUNT(*) > 1
""", conn)

,ticker,date,n


## 5. Build the clean price table

`prices_clean` has one row per fund per official trading day, from listing to the report date.
Missing days are filled with the last available price, and `is_filled = 1` keeps every
filled value traceable.

In [46]:
conn.execute("DROP TABLE IF EXISTS prices_clean")
conn.execute("""
    CREATE TABLE prices_clean AS
    WITH listing AS (
        SELECT ticker, MIN(date) AS first_date
        FROM prices_raw
        GROUP BY ticker
    ),
    grid AS (
        SELECT l.ticker, c.date
        FROM calendar AS c
        JOIN listing AS l ON c.date >= l.first_date
        WHERE c.date <= '2026-09-30'
    )
    SELECT g.ticker,
           g.date,
           COALESCE(
               p.close,
               (SELECT p2.close
                FROM prices_raw AS p2
                WHERE p2.ticker = g.ticker
                  AND p2.date < g.date
                ORDER BY p2.date DESC
                LIMIT 1)
           ) AS close,
           CASE WHEN p.close IS NULL THEN 1 ELSE 0 END AS is_filled
    FROM grid AS g
    LEFT JOIN prices_raw AS p
           ON p.ticker = g.ticker AND p.date = g.date
""")
conn.commit()

pd.read_sql("""
    SELECT ticker,
           COUNT(*)       AS n_days,
           SUM(is_filled) AS n_filled,
           MIN(date)      AS first_date,
           MAX(date)      AS last_date
    FROM prices_clean
    GROUP BY ticker
""", conn)

,ticker,n_days,n_filled,first_date,last_date
0,3DCE.DE,95,0,2026-05-21,2026-09-30
1,3DCH.DE,170,0,2026-02-02,2026-09-30
2,3DGE.DE,361,1,2025-05-02,2026-09-30
3,EUNL.DE,361,2,2025-05-02,2026-09-30


In [47]:
pd.read_sql("SELECT * FROM prices_clean WHERE is_filled = 1", conn)

,ticker,date,close,is_filled
0,3DGE.DE,2025-10-24,6.034000,1
1,EUNL.DE,2025-10-24,109.785004,1
2,EUNL.DE,2026-03-06,113.305000,1


## 6. Performance calculation

Returns are calculated from month-end prices as end / start − 1, for 1M, 3M, YTD, 1Y and
since first price. Funds launched during the year have no YTD or 1Y figure.

Note: 3DCE and 3DCH have less than 12 months of history, so under MiFID rules their
performance would not be shown in a client-facing report.

In [48]:
conn.execute("DROP TABLE IF EXISTS month_end")
conn.execute("""
    CREATE TABLE month_end AS
    SELECT p.ticker, p.date, p.close
    FROM prices_clean AS p
    JOIN (
        SELECT ticker, MAX(date) AS date
        FROM prices_clean
        GROUP BY ticker, substr(date, 1, 7)
    ) AS m
      ON p.ticker = m.ticker AND p.date = m.date
""")
conn.commit()


In [49]:
monthly = pd.read_sql("""
    SELECT ticker,
           substr(date, 1, 7) AS month,
           date,
           close,
           close / LAG(close) OVER (PARTITION BY ticker ORDER BY date) - 1
               AS monthly_return
    FROM month_end
    ORDER BY ticker, date
""", conn)
monthly

,ticker,month,date,close,monthly_return
0,3DCE.DE,2026-05,2026-05-29,5.063600,NaN
1,3DCE.DE,2026-06,2026-06-30,5.084600,0.004147
2,3DCE.DE,2026-07,2026-07-31,5.036000,-0.009558
3,3DCE.DE,2026-08,2026-08-31,5.024000,-0.002383
4,3DCE.DE,2026-09,2026-09-30,4.960300,-0.012679
5,3DCH.DE,2026-02,2026-02-27,5.060000,NaN
6,3DCH.DE,2026-03,2026-03-31,4.950000,-0.021739
7,3DCH.DE,2026-04,2026-04-30,4.970000,0.004040
8,3DCH.DE,2026-05,2026-05-29,5.010000,0.008048
9,3DCH.DE,2026-06,2026-06-30,5.020000,0.001996


In [52]:
perf = pd.read_sql("""
    WITH anchors(period, start_date) AS (          -- 1. Start date of each period
        VALUES ('1M',  '2026-08-31'),
               ('3M',  '2026-06-30'),
               ('YTD', '2025-12-30'),
               ('1Y',  '2025-09-30')
    ),
    end_px AS (                                    -- 2. End point: price on the report date
        SELECT ticker, close AS end_close
        FROM prices_clean
        WHERE date = '2026-09-30'
    ),
    starts AS (                                     -- 3. Each fund's price on each start date
        SELECT p.ticker, a.period, p.close AS start_close
        FROM anchors AS a
        JOIN prices_clean AS p ON p.date = a.start_date
        UNION ALL                                   --    Plus each fund's first available price
        SELECT p.ticker, 'Since first price', p.close
        FROM prices_clean AS p
        WHERE p.date = (SELECT MIN(date) FROM prices_clean AS p2
                        WHERE p2.ticker = p.ticker)
    )
    SELECT s.ticker, s.period, s.start_close, e.end_close,   -- 4. Return = end / start - 1
           e.end_close / s.start_close - 1 AS ret
    FROM starts AS s
    JOIN end_px AS e ON e.ticker = s.ticker
""", conn)

# Pivot: one row per fund, one column per period (full precision kept)
perf_table = perf.pivot(index="ticker", columns="period", values="ret")[
    ["1M", "3M", "YTD", "1Y", "Since first price"]
]

# Format as percentages for display only
perf_table.style.format("{:.2%}", na_rep="–")

period,1M,3M,YTD,1Y,Since first price
ticker,,,,,
3DCE.DE,-1.27%,-2.44%,–,–,-1.19%
3DCH.DE,-1.82%,-2.75%,–,–,-2.36%
3DGE.DE,-0.65%,2.76%,11.67%,17.19%,35.79%
EUNL.DE,1.87%,3.28%,15.53%,20.47%,35.01%


## 7. Reconciliation with published figures

Calculated 3DGE returns are compared with the figures on Robeco's website (as of 30 Sep 2026).

Differences of about 0.25 to 1.4 percentage points go in both directions, which points to a
timing difference rather than a method error: these calculations use Xetra closing prices,
while published figures are based on transaction prices, which reflect markets that close later.
Robeco's factsheet confirms that performance is "net of fees based on transaction prices".

In [55]:
# ===== 1. Published figures (source: Robeco website, 3DGE page, as of 2026-09-30) =====
published = pd.DataFrame({
    "period":        ["1M", "3M", "YTD", "1Y", "Since inception (ann.)"],
    "fund_official": [-0.0114, 0.0206, 0.1192, 0.1583, 0.2529],
    "bm_official":   [-0.0088, 0.0152, 0.1114, 0.1447, 0.2308],
})
published["asof"] = "2026-09-30"
published["source"] = "Robeco website, 3D Global Equity UCITS ETF EUR(H) Acc"
published.to_sql("published_figures", conn, if_exists="replace", index=False)

# ===== 2. Calculated figures, with since-first-price annualised =====
years = (pd.Timestamp("2026-09-30") - pd.Timestamp("2025-05-02")).days / 365.25
mine = perf_table.loc["3DGE.DE", ["1M", "3M", "YTD", "1Y"]].tolist()
mine.append((1 + perf_table.loc["3DGE.DE", "Since first price"]) ** (1 / years) - 1)

# ===== 3. Reconciliation table =====
recon = pd.read_sql("SELECT period, fund_official FROM published_figures", conn)
recon["fund_mine"] = mine
recon["difference"] = recon["fund_mine"] - recon["fund_official"]
recon["explanation"] = [
    "Exchange close vs NAV-based price at start and end",
    "Exchange close vs NAV-based price at start and end",
    "Exchange close vs NAV-based price at start and end",
    "Exchange close vs NAV-based price at start and end",
    "Same, plus different start date (1 May vs 2 May 2025)", 
]

recon.style.format({"fund_official": "{:.2%}",
                    "fund_mine": "{:.2%}",
                    "difference": "{:+.2%}"})

,period,fund_official,fund_mine,difference,explanation
0,1M,-1.14%,-0.65%,+0.49%,Exchange close vs NAV-based price at start and end
1,3M,2.06%,2.76%,+0.70%,Exchange close vs NAV-based price at start and end
2,YTD,11.92%,11.67%,-0.25%,Exchange close vs NAV-based price at start and end
3,1Y,15.83%,17.19%,+1.36%,Exchange close vs NAV-based price at start and end
4,Since inception (ann.),25.29%,24.18%,-1.11%,"Same, plus different start date (1 May vs 2 May 2025)"


## 8. Export to Excel

Exports monthly data, a fund lookup table, performance and the reconciliation to Excel.
Pivot tables, lookups and check formulas were then added by hand in
`fund_report_2026-09_final.xlsx`.

In [56]:
os.makedirs("output", exist_ok=True)

# Check prerequisites
for name in ["perf_table", "recon"]:
    assert name in globals(), f"Run the cell that creates '{name}' first"

# Monthly data
monthly_x = pd.read_sql("SELECT ticker, date, close FROM month_end ORDER BY ticker, date", conn)
monthly_x["month"] = monthly_x["date"].str[:7]
monthly_x["monthly_return"] = monthly_x.groupby("ticker")["close"].pct_change()
monthly_x = monthly_x[["ticker", "month", "date", "close", "monthly_return"]]

# Fund lookup table
funds = pd.DataFrame({
    "ticker":      ["3DGE.DE", "3DCE.DE", "3DCH.DE", "EUNL.DE"],
    "fund_name":   ["Robeco 3D Global Equity EUR(H) Acc",
                    "Robeco 3D EUR Enhanced Index Credits EUR Acc",
                    "Robeco 3D Global Enhanced Index Credits EUR(H) Acc",
                    "iShares Core MSCI World (market reference)"],
    "asset_class": ["Equity", "Fixed income", "Fixed income", "Equity"],
    "isin":        ["IE000WJ7OF21", "IE000PUAKZP8", "IE000A537EY2", "IE00B4L5Y983"],
})

# Write to Excel, one sheet per table
with pd.ExcelWriter("output/fund_report_2026-09.xlsx") as xw:
    monthly_x.to_excel(xw, sheet_name="Data", index=False)
    funds.to_excel(xw, sheet_name="Lookup", index=False)
    perf_table.reset_index().to_excel(xw, sheet_name="Performance", index=False)
    recon.to_excel(xw, sheet_name="Reconciliation", index=False)

print("Saved: output/fund_report_2026-09.xlsx")

Saved: output/fund_report_2026-09.xlsx
